In [ ]:
import torch
torch.__version__

In [ ]:
import math
import torch.nn as nn

In [ ]:
class KVCacheContainer(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads # H
        self.head_dim = embed_dim // num_heads # D

        # Projection
        self.Wk = nn.Linear(self.embed_dim, self.embed_dim)
        self.Wv = nn.Linear(self.embed_dim, self.embed_dim)
        self.Wq = nn.Linear(self.embed_dim, self.embed_dim)

        # KV cache storage (initalize as empty tensors)
        self.k_cache = None #[B, H, S, D]
        self.v_cache = None #[B, H, S, D]

    def forward(self, x: torch.Tensor, is_prefill:bool = True):
        """
        x shape:
          - Prefill phase: [B, S_prefill, Embed_Dim]
          - Decode phase:  [B, 1, Embed_Dim]
        """
        B, S, _ = x.shape
        # x: [B, S, embed_dim] @ [embed_dim, embed_dim] = [B, S, embed_dim]
        # project and restructure to [B, H, S, D]
        q = self.Wq(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.Wk(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.Wv(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)

        #prefill
        if is_prefill:
            self.k_cache = k # [B, H, S, D]
            self.v_cache = v

        #decode
        else:
            # Vectorized concatination along row i.e dim = 2
            self.k_cache = torch.cat([self.k_cache, k], dim=2)
            self.v_cache = torch.cat([self.v_cache, k], dim=2)

        # 4. Compute Scaled Dot-Product Attention using the complete cache
        # q: [B, H, S, D] | k_cache: [B, H, S_total, D] -> scores: [B, H, S, S_total]
        scores = torch.matmul(q, self.k_cache.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # apply causal mask
        if is_prefill:
            mask = torch.tril(torch.ones(S, S, device=x.device)).bool()
            scores = scores.masked_fill(~mask, float('-inf'))

        attn_wgts = torch.softmax(scores, dim=-1)
        output = torch.matmul(attn_wgts, self.v_cache) # [B, H, S, D]

        # Permute and reshape back to original embedding layout [B, S, Embed_Dim]
        output = output.transpose(1, 2).reshape(B, S, self.embed_dim)
        return output

In [ ]:
# --- Verification ---
B, S_prefill, Embed_Dim, Heads = 2, 5, 32, 4
model = KVCacheContainer(embed_dim=Embed_Dim, num_heads=Heads)

# Phase 1: Prefill Pass
prefill_input = torch.randn(B, S_prefill, Embed_Dim)
out_prefill = model(prefill_input, is_prefill=True)
print("Prefill completed. Cache Shape:", model.k_cache.shape) # Expected: [2, 4, 5, 8]

# Phase 2: Autoregressive Step 1 (Decode)
decode_input = torch.randn(B, 1, Embed_Dim)
out_decode = model(decode_input, is_prefill=False)
print("Decode Step 1 completed. New Cache Shape:", model.k_cache.shape) # Expected: [2, 4, 6, 8]

In [ ]:
#